# Florence-2-large — DIMER multi-capability vision-language tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/florence2-vision-language-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/florence2-vision-language-pipeline/blob/main/tutorials/florence2_vision_language_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-florence--community%2FFlorence--2--large-ffcc4d?style=flat)](https://huggingface.co/florence-community/Florence-2-large) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FFlorence--2--large-181717?style=flat&logo=huggingface&logoColor=white)](https://huggingface.co/microsoft/Florence-2-large) [![arXiv](https://img.shields.io/badge/arXiv-2311.06242-b31b1b.svg)](https://arxiv.org/abs/2311.06242)

**Profile:** `MULTI-CAPABILITY`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** prompt-selected vision-language tasks — image captioning, object detection and OCR demonstrated — using the pinned Florence-2-large weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/florence2_vision_language_pipeline/pipeline.py` at revision `9ddb84fc260c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `4271c66b88cdbc05735372ec13b2360108de5317` (~1559 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

Florence-2 is one sequence-to-sequence model whose behaviour is selected by a **task prompt token**: the image is resized to 768 × 768 by the processor (aspect ratio is not preserved), encoded into visual tokens, and the decoder generates text that the processor then parses per task — plain text for captions and OCR, boxes plus labels in input-pixel coordinates for region tasks. Decoding is **deterministic beam search** (`num_beams` = 3 from the snapshot generation config, `do_sample=False`), so a rerun on the same device, dtype and library versions reproduces the same output. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. The pin is the community "official transformers converted checkpoint" (`florence-community/Florence-2-large`), loadable by native `transformers` classes with `trust_remote_code=False`; the original `microsoft/Florence-2-large` snapshot needs remote code and was rejected (see the weight provenance document). What upstream supplies is the model, processor and task-token convention; what the carried pipeline module adds is manifest verification, input validation and ceilings, a fixed per-task output contract, and the `character_error_rate`, `validate_inputs` and `evaluation_report` helpers. The loader also refuses a checkpoint whose weights do not map cleanly onto the native architecture (`output_loading_info` must report no missing, unexpected or mismatched keys).

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, generate a synthetic image with drawn shapes and drawn text (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the ceilings and the task list and validate every demonstrated capability's request into one input manifest, run three capabilities — `<CAPTION>`, `<OD>` and `<OCR>` — through the public API with explicit generation settings, read each capability's output contract correctly, produce an evaluation report that is `sample-sanity` only because OCR can be scored against text the notebook knows and `not-measurable` for the capabilities this repository ships no metric for, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** the other task tokens in `TASKS` are exposed but not demonstrated — `<DETAILED_CAPTION>`, `<MORE_DETAILED_CAPTION>`, `<DENSE_REGION_CAPTION>`, `<REGION_PROPOSAL>`, `<OCR_WITH_REGION>` and `<CAPTION_TO_PHRASE_GROUNDING>` (the only task that takes a `text_input`). Neither this notebook nor the repository provides segmentation of any kind (`<REFERRING_EXPRESSION_SEGMENTATION>`, `<REGION_TO_SEGMENTATION>`), region-to-category or region-to-description prompts, open-vocabulary detection, visual question answering, batched inference, confidence scores for boxes or text, or fine-tuning. The pipeline rejects any task token outside `TASKS`.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float16 there, the dtype the checkpoint ships in); on the model card's CPU smoke the snapshot loaded in 7.1 s, `<CAPTION>` took 4.9 s and `<OD>` 9.0 s on a 256 × 256 drawing, so the three-task default runs in about a minute on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.55 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what beam search is; what a bounding box in pixel coordinates is; what a character error rate measures.
- **Expected output:** a "slow image processor" notice from `transformers` is expected and harmless. A `RuntimeError: checkpoint does not match the native Florence-2 architecture` means the staged weights are not the pinned converted checkpoint.
- **Data:** the default sample is a synthetic image generated in code; BYOD is one image file, gated off by default, any mode (converted to RGB), with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px, resized to 768 × 768 regardless of aspect ratio. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `florence-community/Florence-2-large` snapshot (~1559 MB in total) at revision `4271c66b88cd…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'florence2-vision-language-pipeline',
    'repository_revision': '9ddb84fc260cc9f84ceebb16da26b2c1ba534c0c',
    'embedded_module': 'src/florence2_vision_language_pipeline/pipeline.py',
    'embedded_modules': ['src/florence2_vision_language_pipeline/pipeline.py'],
    'module_sha256': '3d7d6fec855abc35f897d18f5bf707bfa99afd90ca81ab8bcd7ab49abfa9a1d1',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/florence2_vision_language_pipeline/` @ `9ddb84fc260c`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/florence2_vision_language_pipeline/pipeline.py`

In [ ]:
"""Prompt-driven vision-language tasks with the pinned ``florence-community/Florence-2-large`` snapshot.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at the pinned revision, through the native ``transformers`` Florence-2
classes with ``trust_remote_code=False``. Pin history: the original ``microsoft/Florence-2-large`` pin was
rejected on 2026-09-12 because loading it requires executing custom code bundled in the model repository.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "florence-community/Florence-2-large"
MODEL_REVISION = "4271c66b88cdbc05735372ec13b2360108de5317"
MODEL_LICENSE = "mit"
MODEL_KEY = "florence-2-large-community"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

# Task prompts documented upstream for Florence-2; the last one needs a caption as text input.
TASKS_WITHOUT_TEXT = (
    "<CAPTION>",
    "<DETAILED_CAPTION>",
    "<MORE_DETAILED_CAPTION>",
    "<OD>",
    "<DENSE_REGION_CAPTION>",
    "<REGION_PROPOSAL>",
    "<OCR>",
    "<OCR_WITH_REGION>",
)
TASKS_WITH_TEXT = ("<CAPTION_TO_PHRASE_GROUNDING>",)
TASKS = TASKS_WITHOUT_TEXT + TASKS_WITH_TEXT

MAX_IMAGE_SIDE = 4096  # pixels; the processor resizes to 768x768 regardless (preprocessor_config.json)
MAX_TEXT_CHARS = 1000  # characters of caption text accepted for phrase grounding
MAX_NEW_TOKENS = 1024  # hard ceiling for `max_new_tokens` (upstream examples use 1024)
DEFAULT_MAX_NEW_TOKENS = 256
NUM_BEAMS = 3  # snapshot generation_config.json; decoding is deterministic beam search (do_sample=False)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def character_error_rate(reference: str, hypothesis: str) -> float:
    """Character-level Levenshtein distance over reference length; for OCR against a known transcript."""
    if not isinstance(reference, str) or not isinstance(hypothesis, str):
        raise TypeError("reference and hypothesis must be str")
    if not reference:
        return 0.0 if not hypothesis else 1.0
    previous = list(range(len(hypothesis) + 1))
    for i, ref_char in enumerate(reference, 1):
        current = [i]
        for j, hyp_char in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ref_char != hyp_char)))
        previous = current
    return previous[-1] / len(reference)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one task prompt from TASKS; the tasks "
        "in TASKS_WITH_TEXT additionally require a caption string as text_input"
    ),
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "tasks": list(TASKS),
    "tasks_requiring_text_input": list(TASKS_WITH_TEXT),
    "text_input_chars": [1, MAX_TEXT_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": (
        f"positive int; the snapshot's generation_config default is NUM_BEAMS={NUM_BEAMS} and decoding "
        "is deterministic beam search (do_sample=False)"
    ),
    "preprocessing": (
        "image converted to RGB; the processor resizes it to exactly 768x768 (bicubic, ImageNet "
        "mean/std), so aspect ratio is not preserved and nothing is cropped; region outputs are mapped "
        "back to input pixel coordinates. The prompt sent to the model is the task token followed by "
        "text_input when the task takes one."
    ),
}

# Which capability families have an intrinsic metric in this repository, and what the others need.
_OCR_TASK = "<OCR>"
_NEEDS: dict[str, str] = {
    "caption": (
        "reference captions for the same images plus a caption metric (for example CIDEr or SPICE), or "
        "human adequacy ratings; this repository ships neither the references nor a caption metric"
    ),
    "region": (
        "annotated boxes for the same images and the caller's own matching/mean-average-precision code; "
        "Florence-2 emits no per-box score, so there is also nothing to calibrate or threshold"
    ),
    "ocr": (
        "a known transcript for the image, passed as the reference, so character_error_rate can be "
        "computed"
    ),
    "grounding": (
        "annotated boxes for the phrases in the supplied caption and the caller's own matching code; no "
        "grounding metric ships with this repository"
    ),
}
_TASK_FAMILY: dict[str, str] = {
    "<CAPTION>": "caption",
    "<DETAILED_CAPTION>": "caption",
    "<MORE_DETAILED_CAPTION>": "caption",
    "<OD>": "region",
    "<DENSE_REGION_CAPTION>": "region",
    "<REGION_PROPOSAL>": "region",
    "<OCR>": "ocr",
    "<OCR_WITH_REGION>": "region",
    "<CAPTION_TO_PHRASE_GROUNDING>": "grounding",
}
_SCORE_SEMANTICS = (
    "Florence-2 emits no probability or confidence: captions and OCR are plain generated text, and "
    "region tasks return boxes with labels and no per-box score, so there is nothing to threshold or "
    "calibrate. Decoding is deterministic beam search, not a likelihood estimate."
)


def _check_inputs(
    image: Any, task: Any, text_input: Any, max_new_tokens: Any, num_beams: Any
) -> Image.Image:
    """Raise TypeError/ValueError naming the first violated ceiling; return the RGB image.

    ``Florence2Pipeline.run`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if task not in TASKS:
        raise ValueError(f"task must be one of TASKS {TASKS}, got {task!r}")
    if task in TASKS_WITH_TEXT:
        if not isinstance(text_input, str) or not text_input.strip():
            raise ValueError(f"task {task} requires a non-empty text_input")
        if len(text_input) > MAX_TEXT_CHARS:
            raise ValueError(f"text_input exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}: {len(text_input)}")
    elif text_input is not None:
        raise ValueError(f"task {task} takes no text_input")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    if isinstance(num_beams, bool) or not isinstance(num_beams, int) or num_beams < 1:
        raise TypeError("num_beams must be a positive int")
    return image.convert("RGB")


def validate_inputs(
    image: Image.Image,
    task: str = "<CAPTION>",
    text_input: str | None = None,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = NUM_BEAMS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``run`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _check_inputs(image, task, text_input, max_new_tokens, num_beams)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (run takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
            }
        ],
        "task": task,
        "task_requires_text_input": task in TASKS_WITH_TEXT,
        "text_input": text_input,
        "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    reference_text: str | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``result`` is one ``run`` result, or a sequence of them for a multi-capability run. The only
    intrinsic metric in this repository is ``character_error_rate``, and it applies to ``<OCR>``
    when a known transcript is supplied as ``reference_text``; every other capability is
    ``not-measurable`` and the report says what labelled data would make it measurable.
    """
    if not isinstance(result, Mapping):
        subreports = [
            evaluation_report(item, reference_text, sample_kind=sample_kind) for item in result
        ]
        metrics = [
            {**metric, "task": sub["task"]} for sub in subreports for metric in sub["metrics"]
        ]
        return {
            "task": "multi-capability: " + ", ".join(sub["task"] for sub in subreports),
            "score_semantics": _SCORE_SEMANTICS,
            "sample_kind": sample_kind,
            "n_capabilities": len(subreports),
            "metrics": metrics,
            "baselines": [],
            "capabilities": subreports,
            "verdict": "sample-sanity" if metrics else "not-measurable",
            "reason": (
                f"{len(metrics)} capability metric(s) over {len(subreports)} capabilities on one "
                "tutorial sample; sanity evidence, not a benchmark"
                if metrics
                else f"none of the {len(subreports)} demonstrated capabilities has an intrinsic metric here"
            ),
            "needs": "; ".join(dict.fromkeys(sub["needs"] for sub in subreports)),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }
    task = result["task"]
    base = {
        "task": task,
        "score_semantics": _SCORE_SEMANTICS,
        "sample_kind": sample_kind,
        "n_outputs": 1,
        "generation": dict(result.get("generation") or {}),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    family = _TASK_FAMILY.get(task, "region")
    if task == _OCR_TASK and isinstance(reference_text, str) and reference_text.strip():
        reference = reference_text.strip()
        hypothesis = str(result["result"]).strip()
        return {
            **base,
            "metrics": [
                {
                    "id": "character_error_rate",
                    "value": character_error_rate(reference, hypothesis),
                    "reference": reference,
                    "hypothesis": hypothesis,
                    "estimation": "one image against a known transcript, no dispersion estimate",
                }
            ],
            "verdict": "sample-sanity",
            "reason": (
                "one image scored against a transcript the caller already knows; on the synthetic "
                "sample that transcript is text the notebook drew itself, so this is a code-path "
                "check on a rendered font, not an OCR benchmark"
            ),
            "needs": (
                "a labelled OCR corpus from the deployment domain for any generalisable "
                "character-error-rate claim"
            ),
        }
    return {
        **base,
        "metrics": [],
        "verdict": "not-measurable",
        "reason": (
            f"no intrinsic metric exists in this repository for the {family} capability {task}"
            if task != _OCR_TASK
            else "no reference transcript was supplied for the evaluated image"
        ),
        "needs": _NEEDS[family],
    }


@dataclass
class Florence2Pipeline:
    """``_runner(image, prompt, task, max_new_tokens, num_beams)`` -> ``{"text": raw, "parsed": value}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Florence2Pipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Florence2ForConditionalGeneration, Florence2Processor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if resolved_device.startswith("cuda") else torch.float32
        processor = Florence2Processor.from_pretrained(location, **common)
        model, info = Florence2ForConditionalGeneration.from_pretrained(
            location, dtype=dtype, output_loading_info=True, **common
        )
        bad = {k: v for k, v in info.items() if v}
        if bad:
            raise RuntimeError(f"checkpoint does not match the native Florence-2 architecture: {bad}")
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, prompt: str, task: str, max_new_tokens: int, num_beams: int) -> dict:
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(resolved_device, dtype)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            text = processor.batch_decode(generated, skip_special_tokens=False)[0]
            parsed = processor.post_process_generation(text, task=task, image_size=image.size)
            return {"text": text, "parsed": parsed[task]}

        return cls(runner, resolved_device, source)

    def _validate(
        self, image: Any, task: str, text_input: str | None, max_new_tokens: int, num_beams: int
    ) -> None:
        _check_inputs(image, task, text_input, max_new_tokens, num_beams)

    def run(
        self,
        image: Image.Image,
        task: str = "<CAPTION>",
        text_input: str | None = None,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
    ) -> dict[str, Any]:
        """Run one task prompt on one image; ``result`` is the task-parsed value (text, or boxes + labels)."""
        self._validate(image, task, text_input, max_new_tokens, num_beams)
        prompt = task + (text_input or "")
        raw = self._runner(image.convert("RGB"), prompt, task, max_new_tokens, num_beams)
        if not isinstance(raw, dict) or "text" not in raw or "parsed" not in raw:
            raise RuntimeError("runner must return a dict with 'text' and 'parsed'")
        return {
            "task": task,
            "text_input": text_input,
            "result": raw["parsed"],
            "generated_text": str(raw["text"]),
            "image_size": list(image.size),
            "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `12`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4271c66b88cd…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "florence-2-large-community",
  "modelId": "florence-community/Florence-2-large",
  "revision": "4271c66b88cdbc05735372ec13b2360108de5317",
  "files": [
    {
      "path": "README.md",
      "bytes": 13445,
      "sha256": "5662d3853b245397062aa0b1853958f23305e0b9518071293c5154c7eeb415d2"
    },
    {
      "path": "added_tokens.json",
      "bytes": 22430,
      "sha256": "1d75deda84dfa81fb6c09301f3fed00f9695059568bbb1403a6bf299cd84fc37"
    },
    {
      "path": "config.json",
      "bytes": 2396,
      "sha256": "8412483f687f2f71587328a38c6fa70a68d9488f28e90607be3c182641f60f2c"
    },
    {
      "path": "generation_config.json",
      "bytes": 292,
      "sha256": "0251459c49cc358ac033b5d4b8569e61ac22bde5d76be404b97a44cfb33fb12e"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1553541016,
      "sha256": "7715423d6549bf1e71188bdd84f4ac960cc0597886af24a5ef7b66f128660685"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 603,
      "sha256": "1396ec5a0a7adfe1c04fb777b09e8ba753be6dbb5868212ab3c3ef39d91fe031"
    },
    {
      "path": "processor_config.json",
      "bytes": 2264,
      "sha256": "cd0e3bf41a39b1276503fbd273bc03b9afc70d7a15ea681a92a1b4b77f858ee6"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 146627,
      "sha256": "72ff172dc769bc1551b1b4211628ce3271643bc60379e4da45d85a9be9332c39"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3748144,
      "sha256": "3ad7001f773409abe6bba33eac92662611a73d72f459bda2f00d2a221dd31ce4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 197922,
      "sha256": "cb5f80bd9afa767bb1bb798ee5f0a79eae45239b8e72506166b4218175a16723"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 1558929750
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a 512 × 512 white canvas drawn in this cell with a filled red square, a filled blue circle, and the text `DIMER 2026` rendered with Pillow's built-in font — so it needs no download, contains no personal data, and is reproducible from code (no randomness, no seed; its pixel SHA-256 is printed and exported). The drawn text is the **known OCR reference**, the only thing in this notebook that can be scored. A drawing is not a photograph, so every output it produces is smoke/sanity evidence that the code path works, not a quality measurement and not benchmark evidence: the model card's smoke labelled a plain red square `flag`.

BYOD is optional and disabled by default. If your image contains text you know exactly, put it in `OCR_REFERENCE` and Section 7 scores OCR against it; leave it empty otherwise and the OCR capability is reported as `not-measurable` like the other two. The upload stays inside this runtime. Nothing is validated in this cell — the next section hands every demonstrated request to the pipeline's own validation stage, which is the only checker.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
OCR_REFERENCE = ''  # @param {type:"string"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD upload'
    ocr_reference = OCR_REFERENCE.strip() or None
    drawn_square = None
else:
    # Deterministic drawing: a red square, a blue circle and one line of text in the built-in font.
    image = Image.new('RGB', (512, 512), (255, 255, 255))
    draw = ImageDraw.Draw(image)
    drawn_square = (64, 64, 224, 224)
    draw.rectangle(drawn_square, fill=(220, 30, 30))
    draw.ellipse((300, 96, 460, 256), fill=(30, 60, 220))
    ocr_reference = 'DIMER 2026'
    draw.text((96, 360), ocr_reference, fill=(0, 0, 0), font=ImageFont.load_default(size=48))
    image_name = 'synthetic_shapes_text_512'
    sample_kind = 'synthetic (drawn in this cell)'

sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': image_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'ocr_reference': ocr_reference, 'drawn_square': drawn_square, 'pixel_sha256': sample_sha256})

## 5. Validate every capability's request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `run` applies — image type and sides 1..`MAX_IMAGE_SIDE` px, a task token inside `TASKS`, a non-empty `text_input` of at most `MAX_TEXT_CHARS` characters for the tasks in `TASKS_WITH_TEXT` and none for the others, `max_new_tokens` in 1..`MAX_NEW_TOKENS`, and a positive `num_beams` — and returns an **input manifest** naming the schema and ceilings, the input's observed mode and size, the task, whether that task takes a text input, and the exact generation settings. This is a multi-capability notebook, so the cell validates **each** demonstrated capability and writes one combined manifest to `outputs/florence2_vision_language_input_manifest.json`: a top-level record listing the demonstrated tasks, with one per-capability manifest under `capabilities`.

The per-capability input/output contract is printed next to the ceilings, because the three capabilities do not share one: `<CAPTION>` and `<OCR>` return a plain string with no score, while `<OD>` returns `{'bboxes', 'labels'}` in input-pixel coordinates, also with no per-box score. To show what rejection looks like, the cell validates an unsupported task token and records the pipeline's own error message as a finding. **What the pipeline changes about your image:** the processor resizes it to exactly 768 × 768 (bicubic, ImageNet mean/std) — nothing is cropped, but a non-square image is distorted — and region outputs are mapped back to your input's pixel coordinates. The notebook itself does not resize, crop, or subsample.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'NUM_BEAMS': NUM_BEAMS}})
print({'TASKS_WITHOUT_TEXT': TASKS_WITHOUT_TEXT, 'TASKS_WITH_TEXT': TASKS_WITH_TEXT})
CAPABILITIES = {
    '<CAPTION>': {'input': 'image only', 'output': 'result: str (one short caption); no score', 'max_new_tokens': 64},
    '<OD>': {'input': 'image only', 'output': "result: {'bboxes': [[x1, y1, x2, y2], ...] in input pixels, 'labels': [str, ...]}; no per-box score", 'max_new_tokens': DEFAULT_MAX_NEW_TOKENS},
    '<OCR>': {'input': 'image only', 'output': 'result: str (transcribed text, reading order chosen by the model); no score', 'max_new_tokens': 128},
}
for task, contract in CAPABILITIES.items():
    print(task, contract)
manifests = {task: validate_inputs(image, task, max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS, names=[image_name]) for task, contract in CAPABILITIES.items()}
input_manifest = {**manifests['<CAPTION>'], 'task': 'multi-capability: ' + ', '.join(CAPABILITIES), 'tasks': list(CAPABILITIES), 'findings': [], 'capabilities': manifests}
# Demonstrate rejection on a task the pipeline does not expose; the finding is recorded, not swallowed.
try:
    validate_inputs(image, '<REFERRING_EXPRESSION_SEGMENTATION>')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-task-probe', 'task': '<REFERRING_EXPRESSION_SEGMENTATION>', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/florence2_vision_language_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Run the three capabilities

Each call to `run(image, task, *, max_new_tokens=..., num_beams=...)` returns `task`, `result` (the task-parsed value), `generated_text` (the raw decoder output with its special tokens), `image_size`, the `generation` settings actually used (`max_new_tokens`, `num_beams`, `do_sample: False`), device, source and model identity. The settings are printed for every call because they change the output: a caption cut off by `max_new_tokens` is a truncated caption, and a different beam count is a different search.

**Capability A — `<CAPTION>`:** input the image; output one short caption string, no score. **Capability B — `<OD>`:** input the image; output `{bboxes, labels}` in input-pixel coordinates with **no confidence scores** (Florence-2 emits none), so there is no threshold to set and nothing to calibrate. **Capability C — `<OCR>`:** input the image; output the transcribed text as one string, no score. The structural checks below assert the output contract of each capability — that the caption and OCR strings are non-empty text, that the `<OD>` boxes and labels are the same length and lie inside the image, and that every call really used deterministic settings — and raise if any of them fails. They are contract checks, not quality measurements. Outputs are fluent even when wrong — the model can name objects that are not there or transpose characters — and nothing in the output signals it. The printed seconds are measured on this runtime for this one image and include the first-call warm-up.

In [ ]:
import time

results = {}
timings = {}
for task, contract in CAPABILITIES.items():
    started = time.perf_counter()
    results[task] = pipe.run(image, task, max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS)
    timings[task] = round(time.perf_counter() - started, 3)
    print({'task': task, 'seconds': timings[task], 'generation': results[task]['generation'], 'result': results[task]['result']})
caption = results['<CAPTION>']['result']
detections = results['<OD>']['result']
ocr_text = results['<OCR>']['result']
checks = {
    'caption_is_text': isinstance(caption, str) and bool(caption.strip()),
    'od_boxes_and_labels_aligned': isinstance(detections, dict) and len(detections.get('bboxes', [])) == len(detections.get('labels', [])),
    'od_boxes_inside_image': all(0 <= x1 <= x2 <= image.width and 0 <= y1 <= y2 <= image.height for x1, y1, x2, y2 in detections.get('bboxes', [])),
    'ocr_is_text': isinstance(ocr_text, str),
    'deterministic_settings': all(r['generation']['do_sample'] is False and r['generation']['num_beams'] == NUM_BEAMS for r in results.values()),
}
if not all(checks.values()):
    raise RuntimeError(f'capability output failed a sanity check: {checks}')
print({'checks': checks})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report, here over all three capabilities at once: the top level carries the combined verdict and every metric found, and `capabilities` carries one sub-report per task. The **only** intrinsic metric in this repository is `character_error_rate(reference, hypothesis)` — character-level Levenshtein distance divided by the reference length — and it applies to `<OCR>` only when a reference transcript is known. On the default sample that reference is the text the notebook drew, so the figure is a sanity check of the code path on a rendered font, not an OCR benchmark; with no reference the OCR sub-report is `not-measurable` too. `<CAPTION>` and `<OD>` are always `not-measurable` here and the report says what each would need: reference captions plus a caption metric, or annotated boxes plus the caller's own mean-average-precision code. No baseline is reported for any capability, because none is meaningful without labelled data. The report is written to `outputs/florence2_vision_language_evaluation_report.json`.

In [ ]:
report = evaluation_report(list(results.values()), ocr_reference, sample_kind=sample_kind)
with open('outputs/florence2_vision_language_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
for sub in report['capabilities']:
    if sub['verdict'] == 'not-measurable':
        print({'task': sub['task'], 'verdict': sub['verdict'], 'needs': sub['needs']})

## 8. Preview the detections, export results and provenance

The preview draws the `<OD>` boxes and labels onto a copy of the input with Pillow and saves it as `outputs/florence2_vision_language_preview.png` so you can see where the detector placed them; it is a visual aid only — the machine-readable boxes exported alongside it are the outputs intended for downstream use. On the synthetic drawing expect boxes around the shapes with whatever labels the model chose; the model card's smoke called a red square `flag`.

`outputs/florence2_vision_language_result.json` records one entry per capability (task token, parsed `result`, raw `generated_text`, generation settings, seconds), the structural checks, the evaluation report, the input manifest, the ceilings in force, the full list of exposed tasks, the sample identity (name, kind, size, pixel digest, OCR reference, drawn square), the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, Pillow, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
preview = image.convert('RGB').copy()
draw_preview = ImageDraw.Draw(preview)
for (x1, y1, x2, y2), label in zip(detections.get('bboxes', []), detections.get('labels', [])):
    draw_preview.rectangle((x1, y1, x2, y2), outline=(0, 160, 0), width=3)
    draw_preview.text((x1 + 4, y1 + 4), label, fill=(0, 160, 0))
preview.save('outputs/florence2_vision_language_preview.png')
payload = {
    'capabilities': {
        task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'seconds': timings[task]}
        for task, r in results.items()
    },
    'sanity_checks': checks,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'ceilings': {'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'NUM_BEAMS': NUM_BEAMS},
    'tasks_exposed': list(TASKS),
    'sample': {'name': image_name, 'kind': sample_kind, 'width': image.width, 'height': image.height, 'pixel_sha256': sample_sha256, 'ocr_reference': ocr_reference, 'drawn_square': drawn_square},
    'preview_file': 'outputs/florence2_vision_language_preview.png',
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'source': pipe.source,
        'dtype': 'float16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/florence2_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The three outputs are generated text parsed per task: a caption string, boxes with labels and **no confidence scores**, and an OCR string. None of them carries a probability, and the only quantity the notebook scores is OCR against text it knows — on the default sample the text it drew itself, which makes the character error rate a code-path sanity check on a rendered font, not an OCR benchmark; captions and detections are reported `not-measurable` because this repository ships no metric for them, and they need human judgement or annotated references plus the caller's own evaluation code. Every output is fluent whether or not it is right: the model can describe objects that are not there, label a plain square as a `flag`, or transpose characters, and nothing in the output signals it. The image is squashed to 768 × 768 before encoding, so thin or off-aspect content is distorted; region outputs are mapped back to input pixels. Decoding is deterministic beam search (3 beams, no sampling) on a fixed device and dtype; CPU float32 and CUDA float16 can produce different text. The repository exposes the nine task tokens in `TASKS` — including phrase grounding, which takes a caption as `text_input` — and nothing else: no segmentation of any kind, no region-to-category or region-to-description prompts, no open-vocabulary detection, no VQA, no batching.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated requests against the enforced ceilings, execute the public pipeline path for three task tokens with explicit generation settings, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, caption, detection or OCR quality on any domain, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/florence-2-large-community/` and rerun Section 3. `RuntimeError: checkpoint does not match the native Florence-2 architecture` in Section 3: the staged weights are not the pinned converted checkpoint — re-stage. A `ValueError` naming `MAX_IMAGE_SIDE` in Section 5: resize the BYOD image and rerun from Section 4. A truncated caption or OCR string: raise that task's `max_new_tokens` in Section 5 (up to `MAX_NEW_TOKENS`). A "slow image processor" notice from `transformers` is expected and harmless.

**Next experiments.** Upload a photograph with readable signage and supply `OCR_REFERENCE` to see how the character error rate behaves on real text; add `<CAPTION_TO_PHRASE_GROUNDING>` to `CAPABILITIES` with the caption from Capability A as its `text_input` and compare its boxes with `<OD>`; run the same image on a CUDA runtime and diff the float16 outputs against the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and pin history: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/docs/WEIGHTS.md
- Pinned converted checkpoint: https://huggingface.co/florence-community/Florence-2-large
- Original weights and licence: https://huggingface.co/microsoft/Florence-2-large
- Florence-2 paper: https://arxiv.org/abs/2311.06242
- Transformers Florence-2 documentation: https://huggingface.co/docs/transformers/model_doc/florence2